❯ /home/gao/projects_2026H2/5_mouse-rRNA/SortMeRNA_issue/06_kraken2_contamination_check.sh 请告诉我检查的是rRNA reads , hais non-RNA reas , 还有是不是检查的无法mapping到基因组上的reads？

  Read 1 file

──────────────────────────────────────────────────────── 1 new message ─────────────────────────────────────────────────────────

三个问题，逐一回答：

检查的是什么 reads？

脚本分两层过滤，Kraken2 检查的是既不是 rRNA、又没有比对到小鼠基因组的 reads：

| :--- | :---: |
| 输入 FASTQ | non_rRNA.fastq.gz — SortMeRNA 之后剩下的，即已排除 rRNA 的 reads |
| 再过滤 | 从上面的 FASTQ 中，减去 STAR BAM 里有比对记录的 read ID（comm -23）|
| 送给 Kraken2 的 | non-rRNA + 未比对到 GRCm39 的 reads |

所以：不是 rRNA reads，是 non-rRNA reads 中无法 mapping 到小鼠基因组的那部分。

---
具体逻辑（第 56–64 行）：

BAM (STAR Aligned.out.bam)
  └─ samtools view -F 0x100   →  mapped_ids.txt   (已比对上的 read 名)

non_rRNA FASTQ
  └─ awk 提取 read 名          →  all_ids.txt      (全部 non-rRNA read 名)

comm -23 all_ids mapped_ids   →  unmapped_ids.txt (non-rRNA 且未比对的)

seqtk subseq 按名提取两端     →  unmapped_1.fq / unmapped_2.fq
                                        ↓
                                    Kraken2

-F 0x100 排除的是 secondary alignment（flag 256），目的是避免同一 read 被计入两次。STAR 默认不把 unmapped reads 写入 BAM，所以 mapped_ids.txt 里确实只有比对上的 reads。

---
问题背景（脚本注释说明）：mouse_46 和 mouse_29 这两个样品经过 SortMeRNA 去 rRNA 后，剩余 reads 的 STAR 比对率极低，minimap2 宽松模式也比不上去，只有约 1% 是 adapter read-through。Kraken2 的目的是查清楚这批"非鼠、非 rRNA"的 reads 究竟来自什么物种（细菌？病毒？真菌污染？）。


宏基因组分类深度解读报告 (系统性异常批次分析)

本报告针对样本 mouse_46 及 mouse_29 的 Kraken2 物种分类结果进行系统性对比与解析。
通过交叉比对两份独立样本的数据，我们发现其具备完全相同的“异常指纹”。这高度提示该测序批次面临系统性的建库污染（Kit-ome）、操作人员核酸污染以及细胞系支原体感染，同时证实了当前使用的 Kraken2 数据库严重缺失小鼠（Mus musculus）本底参考基因组。

一、 多样本宏观分类数据大盘

经过 Kraken2 数据库比对，两个样本展现出高度一致的分布规律：

分类大类 (Category)

mouse_46 Reads (丰度)

mouse_29 Reads (丰度)

核心发现与批次特征

总 Reads 规模

$4,842,956$

$1,172,205$

两个样本数据规模相差约 4 倍，但组分分布几乎平行。

未分类 (Unclassified)

$4,180,799$ ($86.33\%$)

$997,748$ ($85.12\%$)

极高的未比对率，证实数据库未收录宿主（小鼠/CHO）基因组。

已分类 (Classified)

$662,157$ ($13.67\%$)

$174,457$ ($14.88\%$)

-

├── 真核生物 (Eukaryota)

$563,163$ ($11.63\%$)

$145,217$ ($12.39\%$)

$100\%$ 被错分/污染为人类（Homo sapiens）。

├── 细菌 (Bacteria)

$94,084$ ($1.94\%$)

$28,578$ ($2.44\%$)

包含一致的致病菌与细胞系污染指示菌。

└── 病毒 (Viruses)

$346$ ($0.01\%$)

$94$ ($0.01\%$)

极低丰度的噬菌体和反转录病毒（如内源性逆转录病毒残留）。

二、 系统性异常信号解析 (Batch Red Flags)

由于在两份测序深度截然不同的样本中，发现了按比例缩放的同源污染，这标志着系统性问题：

1. 宿主屏蔽与系统性人类 DNA 污染 (丰度稳定在 $11\% \sim 12\%$)

现象：在两份名为“mouse”的样本中，没有任何小鼠的分类节点。所有比对上的真核序列（在 mouse_46 中是 $56万$ 条，在 mouse_29 中是 $14.5万$ 条）全部精确指向了 Homo sapiens。

生信诊断：
这确诊了我们之前的猜测。您的 Kraken2 数据库（极大概率是纯微生物 + 人类构成的 MiniKraken）不包含小鼠参考基因组。真正的小鼠 Reads 全部落入了 $85\%+$ 的 Unclassified 盲区。而那些比对到人的序列，一部分源自人鼠高度保守的同源序列被强行错误归类，另一部分则是实验环境真实的系统性人源气溶胶/接触污染。

2. 实锤警告：细胞系支原体污染 (Mycoplasma)

现象：两份样本均检出 猪鼻支原体（Mesomycoplasma hyorhinis）。mouse_46 包含 $3,898$ 条，mouse_29 包含 $119$ 条。

生信诊断：
在两个不同样本中同时扫出相同的特定支原体，基本可以给该批次细胞系的“污染状态”定罪。M. hyorhinis 是细胞房最臭名昭著的污染源，它会改变宿主细胞的代谢途径和基因表达。该批次如果用于后续的转录组 (RNA-seq) 或单细胞分析，其细胞生理状态的数据将面临严重的科学严谨性挑战。

3. 建库试剂盒/水源的本底污染 ("Kit-ome")

现象：环境微生物 弗氏弧菌（Vibrio furnissii） 在 mouse_46 中高达 $12,019$ 条，在 mouse_29 中检出 $4,114$ 条。此外，两份样本均伴生了大量皮肤/环境常见菌（如 Cutibacterium acnes, 枯草芽孢杆菌群等）。

生信诊断：
非水体样本出现如此高比例、且跨样本浓度稳定的水生弧菌，这几乎肯定是建库试剂盒污染（Kit-ome）或实验室超纯水系统污染。这是低生物量（Low-biomass）宏基因组测序中极其经典的假阳性背景噪音。

三、 针对您 70 小时高压环境的下一步行动建议

作为生物信息学家，您现在有充足的证据向实验团队（Wet Lab）和生信分析流（Dry Lab）提出以下补救措施：

Step 1: 湿实验（Wet Lab）紧急预警

拿着这份双样本高度一致的报告，立刻要求实验团队对培养箱中的该批次小鼠/CHO细胞进行支原体 PCR 验证。同时，检查核酸提取使用的超纯水和缓冲液是否存在弧菌本底污染。

Step 2: 宿主序列打捞（拯救您的 85% 数据）

放弃 Kraken2 的分析，用您定制的 GRCm39 小鼠全基因组索引，通过底层比对器直接打捞真实的宿主数据。

请在 Positron 终端中，利用 mamba activate R44_RNA 激活您的环境，运行以下命令（以 mouse_29 为例）：

# 务必确认您已经执行过 bwa-mem2 index 命令建立了索引
bwa-mem2 mem -t 64 /Work_bio/references/Mus_musculus/GRCm39/mouse_gencode_M35/GRCm39.primary_assembly.genome.fa mouse_29_R1.fq.gz mouse_29_R2.fq.gz | samtools flagstat -


如果 samtools flagstat 报告显示比对率超过 $85\%$，则说明测序数据基本正常，所谓的“人源污染”主要是因为 Kraken 数据库中缺失小鼠导致的算法迫选（Force-assignment）假象。

四、 附录：数据统计表格 Raw Markdown 源码

为方便您复制到 Positron/VS Code 中进行汇报，以下为多样本数据大盘的无格式损失源码：

| 分类大类 (Category) | `mouse_46` Reads (丰度) | `mouse_29` Reads (丰度) | 核心发现与批次特征 | 
| :--- | :--- | :--- | :--- | 
| **总 Reads 规模** | $4,842,956$ | $1,172,205$ | 两个样本数据规模相差约 4 倍，但组分分布几乎平行。 | 
| **未分类 (Unclassified)** | $4,180,799$ ($86.33\%$) | $997,748$ ($85.12\%$) | 极高的未比对率，证实数据库未收录宿主（小鼠/CHO）基因组。 | 
| **已分类 (Classified)** | $662,157$ ($13.67\%$) | $174,457$ ($14.88\%$) | - | 
| ├── **真核生物 (Eukaryota)** | $563,163$ ($11.63\%$) | $145,217$ ($12.39\%$) | **$100\%$ 被错分/污染为人类（*Homo sapiens*）**。 | 
| ├── **细菌 (Bacteria)** | $94,084$ ($1.94\%$) | $28,578$ ($2.44\%$) | 包含一致的致病菌与细胞系污染指示菌。 | 
| └── **病毒 (Viruses)** | $346$ ($0.01\%$) | $94$ ($0.01\%$) | 极低丰度的噬菌体和反转录病毒（如内源性逆转录病毒残留）。 | 
